***

Preparing Workspace

***

In [ ]:


## Importing packages ---

import numpy as np
import pandas as pd
import os
import tqdm
import urllib.request, json


## Setting file paths ---

user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_code    = os.path.join(path_git, 'Data', 'Census')
    path_config0 = os.path.join(path_git , 'config')
    path_config  = os.path.join(path_code, 'config')


## User defined functions ---

exec(open(os.path.join(path_config0, 'Functions.py')).read())



***

Importing

***

In [ ]:


## ACS5 ---
## Organize list of all variables from all years into one nice table
# Set which years to import
# Use urllib package to make request to Census API (requesting for table of all variables sampled on the given year)
# Do some cleaning, reshaping, etc... to make nice for use in data pipelines

years_to_import = sequence(2020, 2020, 1)
list_df = []

for year in tqdm(years_to_import):
    with urllib.request.urlopen(f"https://api.census.gov/data/{year}/acs/acs5/variables.json") as url:
        dict_acs = json.load(url)
    
    df_vars = pd.DataFrame.from_dict(dict_acs['variables']).T.reset_index().rename(columns = {'index':'ID'})
    
    df_vars = df_vars[['group', 'ID', 'attributes', 'label', 'concept']].rename(columns = {'group':'Table'})
    df_vars['Label_clean'] = df_vars['label'].str.replace('Estimate!!', '')
    df_vars['Label_clean'] = df_vars['Label_clean'].str.replace('!!', ' ')
    df_vars['Label_clean'] = df_vars['Label_clean'].str.replace(':', '')
    df_vars['Year'] = year

    list_df.append(df_vars)

df_acs5 = pd.concat(list_df)
df_acs5 = df_acs5.sort_values(['Year', 'Table', 'ID'], ascending = [False, True, True])
df_acs5 = df_acs5.reset_index(drop=True)
print(df_acs5.shape)
df_acs5.head()



In [ ]:


## ACS1 ---
## Organize list of all variables from all years into one nice table
# Set which years to import
# Use urllib package to make request to Census API (requesting for table of all variables sampled on the given year)
# Do some cleaning, reshaping, etc... to make nice for use in data pipelines

years_to_import = sequence(2005, 2023, 1)
years_to_import.remove(2020)
list_df = []

for year in tqdm(years_to_import):
    with urllib.request.urlopen(f"https://api.census.gov/data/{year}/acs/acs1/variables.json") as url:
        dict_acs = json.load(url)
    
    df_vars = pd.DataFrame.from_dict(dict_acs['variables']).T.reset_index().rename(columns = {'index':'ID'})
    
    df_vars = df_vars[['group', 'ID', 'attributes', 'label', 'concept']].rename(columns = {'group':'Table'})
    df_vars['Label_clean'] = df_vars['label'].str.replace('Estimate!!', '')
    df_vars['Label_clean'] = df_vars['Label_clean'].str.replace('!!', ' ')
    df_vars['Label_clean'] = df_vars['Label_clean'].str.replace(':', '')
    df_vars['Year'] = year

    list_df.append(df_vars)

df_acs1 = pd.concat(list_df)
df_acs1 = df_acs1.sort_values(['Year', 'Table', 'ID'], ascending = [False, True, True])
df_acs1 = df_acs1.reset_index(drop=True)
print(df_acs1.shape)
df_acs1.head()



In [ ]:


df_acs1 = df_acs1[df_acs1['Year'] != 2020]
df_acs5 = df_acs5[df_acs5['Year'] == 2020]

df_acs = pd.concat([df_acs1, df_acs5])
df_acs = df_acs.sort_values(['Table', 'Year', 'ID'], ascending = [True, False, True])
df_acs = df_acs.reset_index(drop=True)

display(df_acs.head(), df_acs.tail())



In [ ]:


df_config = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsm'), sheet_name='ACS')
df_config = df_config[['Table', 'ID', 'Indicator Name', 'Include', 'Variable', 'Sort', 'Race_Ethnicity']]

df_acs = df_acs.merge(df_config, on=['Table', 'ID'], how='left')
df_acs.head()



***

Exporting

***

In [ ]:


## Exporting to Git ---

workbook_name = 'Census Variable Tables_ACS.xlsx'
path_sdl = r'I:/Projects/Josh/Regional Monitoring'

# df_acs.to_excel(os.path.join(path_sdl, workbook_name), sheet_name = 'ACS', index=False)

